# Tutorial 4: Molecule (Drug) Embeddings

This notebook demonstrates how to **generate**, **load**, **analyze**, and
**annotate** small-molecule embeddings with `embpy`.

### What this tutorial covers

1. **Generate embeddings** for 5 example molecules using `BioEmbedder`
2. **Load pre-computed embeddings** from multiple models into a single AnnData
3. **Heatmaps** of embedding values
4. **Cross-model correlation** -- how similar are embedding spaces?
5. **Leiden clustering + UMAP** per model
6. **Full annotation** with `MoleculeAnnotator` (RDKit, ChEMBL, ChEBI, KEGG, PubChem, UniChem)
7. **Annotation enrichment per cluster** -- linking molecular properties to embedding structure

### Available molecule models

| Model | Key | Dim | Type |
|---|---|---|---|
| ChemBERTa-2 (MTR) | `chemberta2MTR` | 384 | Transformer |
| ChemBERTa-2 (MLM) | `chemberta2MLM` | 768 | Transformer |
| MolFormer | `molformer_base` | 768 | Transformer |
| Morgan Fingerprint | `morgan_fp` | 1024 | Classical |
| RDKit Descriptors | `rdkit_fp` | 200 | Classical |
| MiniMol | `minimol` | 512 | GNN |
| MHG-GNN | `mhg_gnn` | 64 | Hypergraph GNN |
| MolE | `mole` | 768 | Graph Transformer |

In [3]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

import numpy as np
import pandas as pd
import anndata as ad
import scanpy as sc
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook", font_scale=1.1)
%matplotlib inline

---
## 1. Generate embeddings for 5 molecules

We use `BioEmbedder` to embed a handful of well-known drugs. This
demonstrates the full resolution + inference pipeline.

In [4]:
from embpy.embedder import BioEmbedder

embedder = BioEmbedder(device="auto")
print(f"Device: {embedder.device}")

: 

: 

: 

In [ ]:
demo_molecules = {
    "Aspirin":     "CC(=O)Oc1ccccc1C(O)=O",
    "Caffeine":    "Cn1c(=O)c2c(ncn2C)n(C)c1=O",
    "Ibuprofen":   "CC(C)Cc1ccc(cc1)C(C)C(O)=O",
    "Penicillin":  "CC1(C)SC2C(NC(=O)Cc3ccccc3)C(=O)N2C1C(O)=O",
    "Dexamethasone": "CC1CC2C3CCC4=CC(=O)C=CC4(C)C3(F)C(O)CC2(C)C1(O)C(=O)CO",
}

DEMO_MODELS = ["chemberta2MTR", "chemberta2MLM", "molformer_base"]

demo_results = {}
for model in DEMO_MODELS:
    print(f"\n--- {model} ---")
    for name, smi in demo_molecules.items():
        emb = embedder.embed_molecule(smi, model=model)
        demo_results[(model, name)] = emb
        print(f"  {name:15s}  dim={emb.shape[0]}")

In [ ]:
import embpy.pl as epl

demo_adata = ad.AnnData(
    obs=pd.DataFrame({"molecule": list(demo_molecules.keys())},
                      index=list(demo_molecules.keys())),
)
for model in DEMO_MODELS:
    demo_adata.obsm[model] = np.stack([demo_results[(model, n)] for n in demo_molecules])

for model in DEMO_MODELS:
    epl.plot_similarity_heatmap(
        adata=demo_adata, obsm_key=model, metric="cosine",
        labels=list(demo_molecules.keys()),
        title=f"Cosine similarity ({model})",
        figsize=(5, 4), annot=True, fmt=".2f",
    )
    plt.show()

---
## 2. Load pre-computed embeddings

We have pre-computed embeddings for ~142k molecules across 5 models.
For interactive analysis we subsample 2000 molecules shared across all
models. Each model's embeddings are stored in `adata.obsm`.

In [ ]:
DATA_DIR = "../../data/embeddings/drug_embeddings"

N_SUBSAMPLE = 2000
SEED = 42

In [ ]:
def load_embedding_csv(path, smiles_col=None, drop_cols=None):
    """Load an embedding CSV and return (smiles_array, embedding_matrix)."""
    df = pd.read_csv(path)
    if smiles_col is None:
        for c in ["smiles", "Unnamed: 0"]:
            if c in df.columns:
                smiles_col = c
                break
    smiles = df[smiles_col].values
    drop = [c for c in df.columns if not c[0].isdigit() and not c.startswith("e_")]
    if drop_cols:
        drop += drop_cols
    emb_df = df.drop(columns=[c for c in drop if c in df.columns])
    return smiles, emb_df.values.astype(np.float32)


print("Loading RDKit descriptors...")
smiles_rdkit, X_rdkit = load_embedding_csv(f"{DATA_DIR}/rdkit/descriptors_200.csv")
print(f"  {X_rdkit.shape[0]:,} molecules x {X_rdkit.shape[1]} dims")

print("Loading Morgan fingerprints...")
smiles_morgan, X_morgan = load_embedding_csv(f"{DATA_DIR}/morgan_fingerprint/fingerprint_1024.csv")
print(f"  {X_morgan.shape[0]:,} molecules x {X_morgan.shape[1]} dims")

print("Loading ChemBERTa-2 MTR (mean)...")
smiles_mtr, X_mtr = load_embedding_csv(f"{DATA_DIR}/chemberta2MTR/embeddings_mean.csv")
print(f"  {X_mtr.shape[0]:,} molecules x {X_mtr.shape[1]} dims")

print("Loading ChemBERTa-2 MLM (mean)...")
smiles_mlm, X_mlm = load_embedding_csv(f"{DATA_DIR}/chemberta2MLM/embeddings_mean.csv")
print(f"  {X_mlm.shape[0]:,} molecules x {X_mlm.shape[1]} dims")

print("Loading MolFormer (mean)...")
smiles_molf, X_molf = load_embedding_csv(f"{DATA_DIR}/molformer_base/embeddings_mean.csv")
print(f"  {X_molf.shape[0]:,} molecules x {X_molf.shape[1]} dims")

In [ ]:
common = set(smiles_rdkit) & set(smiles_mtr) & set(smiles_mlm) & set(smiles_molf)
print(f"Molecules in common across all models: {len(common):,}")

rng = np.random.default_rng(SEED)
sample_smiles = rng.choice(list(common), size=min(N_SUBSAMPLE, len(common)), replace=False)
print(f"Subsampled {len(sample_smiles):,} molecules for analysis")

In [ ]:
def reindex_to_smiles(smiles_arr, X, target_smiles):
    """Reorder rows of X so they match target_smiles ordering."""
    idx_map = {s: i for i, s in enumerate(smiles_arr)}
    indices = [idx_map[s] for s in target_smiles]
    return X[indices]

adata = ad.AnnData(
    obs=pd.DataFrame({"smiles": sample_smiles}, index=[f"mol_{i}" for i in range(len(sample_smiles))]),
)

adata.obsm["X_rdkit"] = reindex_to_smiles(smiles_rdkit, X_rdkit, sample_smiles)
adata.obsm["X_morgan"] = reindex_to_smiles(smiles_morgan, X_morgan, sample_smiles)
adata.obsm["X_chemberta_mtr"] = reindex_to_smiles(smiles_mtr, X_mtr, sample_smiles)
adata.obsm["X_chemberta_mlm"] = reindex_to_smiles(smiles_mlm, X_mlm, sample_smiles)
adata.obsm["X_molformer"] = reindex_to_smiles(smiles_molf, X_molf, sample_smiles)

print(adata)
print("\nobsm keys:", list(adata.obsm.keys()))
for k in adata.obsm:
    print(f"  {k}: {adata.obsm[k].shape}")

---
## 3. Embedding heatmaps

Visualize the raw embedding matrices as clustermaps. Hierarchical
clustering of both molecules (rows) and dimensions (columns) reveals
internal structure of each embedding space.

In [ ]:
import embpy.pl as epl

MODELS = {
    "X_rdkit":          "RDKit (200d)",
    "X_morgan":         "Morgan FP (1024d)",
    "X_chemberta_mtr":  "ChemBERTa MTR (384d)",
    "X_chemberta_mlm":  "ChemBERTa MLM (768d)",
    "X_molformer":      "MolFormer (768d)",
}

model_keys = list(MODELS.keys())
model_labels = list(MODELS.values())

for key, label in MODELS.items():
    epl.embedding_clustermap(adata, obsm_key=key, n_obs=200, title=f"Embedding heatmap: {label}")
    plt.show()

### Embedding dimension distributions and norms

How are values distributed within each embedding space? Violin plots
show the first 10 dimensions; the box plot compares L2 norms across models.

In [ ]:
epl.embedding_distributions(adata, obsm_keys=model_keys, n_dims=10)
plt.show()

In [ ]:
epl.embedding_norms(adata, obsm_keys=model_keys)
plt.show()

---
## 4. Cross-model correlation

How much do different embedding spaces agree? For each pair of models
we compute the **pairwise cosine similarity matrix** within each model,
then correlate those similarity matrices (Mantel-like comparison).

In [ ]:
epl.cross_model_similarity(
    adata, obsm_keys=model_keys, method="cosine_correlation",
    labels=model_labels, figsize=(8, 6),
)
plt.show()

In [ ]:
epl.cross_model_similarity(
    adata, obsm_keys=model_keys, method="knn",
    labels=model_labels, figsize=(8, 6), cmap="YlOrRd", k=20,
)
plt.show()

In [ ]:
epl.knn_overlap(
    adata, obsm_keys=model_keys, k=20, figsize=(8, 6),
)
plt.show()

### Per-model similarity and distance heatmaps

Zoom into a single model's pairwise structure with `correlation_matrix`
and `distance_heatmap` (showing the first 50 molecules for readability).

In [ ]:
adata_50 = adata[:50].copy()

epl.correlation_matrix(
    adata_50, obsm_key="X_chemberta_mtr", method="pearson",
    title="Pearson correlation (ChemBERTa MTR, 50 molecules)",
)
plt.show()

epl.distance_heatmap(
    adata_50, obsm_key="X_chemberta_mtr", metric="cosine",
    title="Cosine distance (ChemBERTa MTR, 50 molecules)",
)
plt.show()

### Cross-embedding scatter

Do two models agree on which molecules are similar? Each dot represents
a pair of molecules; axes show their cosine similarity in each space.

---
## 5. Leiden clustering + UMAP

For each model we compute a neighbor graph, run Leiden clustering, and
project to 2-D with UMAP.

In [ ]:
from embpy.tl.clustering import leiden
from embpy.tl.dimred import compute_umap

for key, label in MODELS.items():
    cluster_key = f"leiden_{key}"
    leiden(adata, obsm_key=key, resolution=0.5, n_neighbors=15, key_added=cluster_key)
    compute_umap(adata, obsm_key=key, n_neighbors=15)
    print(f"{label}: {adata.obs[cluster_key].nunique()} Leiden clusters")

In [ ]:
for key, label in MODELS.items():
    epl.leiden_overview(
        adata, obsm_key=key, resolution=0.5,
        plots=["umap_cluster", "cluster_sizes"],
        figsize=(12, 5),
    )
    plt.show()

### Cluster agreement across models

Do different models produce similar groupings? We measure cluster
agreement with the Adjusted Rand Index (ARI).

In [ ]:
epl.cross_model_similarity(
    adata, obsm_keys=model_keys, method="ari",
    labels=model_labels, figsize=(8, 6), cmap="YlGnBu", resolution=0.5,
)
plt.show()

---
## 6. Molecule annotation

We annotate a subset of molecules using `MoleculeAnnotator`, which
queries six public databases:

| Source | Annotation |
|---|---|
| RDKit (local) | Physicochemical properties (MW, LogP, TPSA, HBD/HBA, rotatable bonds, ...) |
| ChEMBL | Bioactivities (IC50, Ki, EC50), target proteins, mechanism of action |
| ChEBI | Ontological roles (e.g. "anti-inflammatory", "kinase inhibitor") |
| KEGG | Metabolic and signaling pathway annotations |
| PubChem + UniChem | Cross-database identifiers (CID, ChEMBL ID, KEGG, DrugBank, ...) |
| PubChem | Disease associations |

Since API calls are rate-limited, we annotate a manageable subset of
100 molecules and map the scalar properties back to the full AnnData.

In [ ]:
from embpy.resources.molecule_annotator import MoleculeAnnotator

annotator = MoleculeAnnotator(rate_limit_delay=0.25)

In [ ]:
N_ANNOTATE = 100

rng_ann = np.random.default_rng(SEED)
ann_indices = rng_ann.choice(adata.n_obs, size=N_ANNOTATE, replace=False)
ann_smiles = adata.obs["smiles"].iloc[ann_indices].values
print(f"Annotating {N_ANNOTATE} molecules...")

In [ ]:
annotations = []
for i, smi in enumerate(ann_smiles):
    if (i + 1) % 20 == 0:
        print(f"  {i + 1}/{N_ANNOTATE}")
    ann = annotator.annotate(smi, categories="all")
    ann["smiles"] = smi
    annotations.append(ann)
print(f"Done. Annotated {len(annotations)} molecules.")

In [ ]:
physchem_records = []
for ann in annotations:
    pc = ann.get("physicochemical", {})
    pc["smiles"] = ann["smiles"]
    pc["n_bioactivities"] = len(ann.get("bioactivities", []))
    pc["n_targets"] = len(ann.get("targets", []))
    pc["n_moa"] = len(ann.get("mechanism_of_action", []))
    pc["n_chebi_roles"] = len(ann.get("chebi_roles", []))
    pc["n_kegg_pathways"] = len(ann.get("kegg_pathways", []))
    pc["n_diseases"] = len(ann.get("diseases", []))
    pc["has_chembl"] = 1 if ann.get("cross_references", {}).get("chembl_id") else 0
    pc["has_drugbank"] = 1 if "drugbank" in str(ann.get("cross_references", {})).lower() else 0
    physchem_records.append(pc)

df_ann = pd.DataFrame(physchem_records).set_index("smiles")
print(f"Annotation table: {df_ann.shape[0]} molecules x {df_ann.shape[1]} features")
df_ann.head()

In [ ]:
ann_map = df_ann.to_dict(orient="index")

NUMERIC_PROPS = [
    "molecular_weight", "logp", "tpsa", "hbd", "hba",
    "rotatable_bonds", "num_rings", "num_aromatic_rings",
    "fraction_sp3", "num_heavy_atoms",
    "n_bioactivities", "n_targets", "n_moa",
    "n_chebi_roles", "n_kegg_pathways", "n_diseases",
]

for prop in NUMERIC_PROPS:
    vals = []
    for smi in adata.obs["smiles"]:
        if smi in ann_map and prop in ann_map[smi]:
            vals.append(ann_map[smi][prop])
        else:
            vals.append(np.nan)
    adata.obs[prop] = vals

n_annotated = adata.obs["molecular_weight"].notna().sum()
print(f"Mapped annotation properties to adata.obs: {n_annotated}/{adata.n_obs} molecules annotated")

### Physicochemical property distributions

In [ ]:
physchem_props = ["molecular_weight", "logp", "tpsa", "hbd", "hba",
                  "rotatable_bonds", "num_rings", "fraction_sp3"]

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for ax, prop in zip(axes.flat, physchem_props):
    vals = adata.obs[prop].dropna()
    ax.hist(vals, bins=30, edgecolor="white", alpha=0.8, color="steelblue")
    ax.set_title(prop.replace("_", " ").title(), fontsize=11)
    ax.set_xlabel("")
plt.suptitle("Physicochemical property distributions (annotated subset)", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

### Annotation counts

In [ ]:
count_props = ["n_bioactivities", "n_targets", "n_moa", "n_chebi_roles", "n_kegg_pathways", "n_diseases"]

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, prop in zip(axes.flat, count_props):
    vals = adata.obs[prop].dropna()
    ax.hist(vals, bins=20, edgecolor="white", alpha=0.8, color="coral")
    ax.set_title(prop.replace("n_", "# ").replace("_", " ").title(), fontsize=11)
    nonzero = (vals > 0).sum()
    ax.annotate(f"{nonzero}/{len(vals)} have >0", xy=(0.95, 0.95), xycoords="axes fraction",
                ha="right", va="top", fontsize=9, color="gray")
plt.suptitle("Database annotation counts per molecule", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

### Physicochemical property correlation matrix

In [ ]:
all_props = physchem_props + count_props
df_props = adata.obs[all_props].dropna()

fig, ax = plt.subplots(figsize=(10, 8))
prop_corr = df_props.corr()
mask = np.triu(np.ones_like(prop_corr, dtype=bool), k=1)
sns.heatmap(
    prop_corr, mask=mask, annot=True, fmt=".2f", cmap="RdBu_r",
    center=0, vmin=-1, vmax=1, ax=ax, square=True,
    linewidths=0.5, linecolor="white",
)
ax.set_title("Correlation between molecular properties and annotation counts", fontsize=13)
plt.tight_layout()
plt.show()

---
## 7. Annotation enrichment per Leiden cluster

Do Leiden clusters capture meaningful chemical diversity? For each model,
we compare physicochemical properties and database annotation counts
across clusters.

In [ ]:
REFERENCE_MODEL = "X_chemberta_mtr"
CLUSTER_KEY = f"leiden_{REFERENCE_MODEL}"

annotated_mask = adata.obs["molecular_weight"].notna()
adata_ann = adata[annotated_mask].copy()
print(f"Annotated molecules: {adata_ann.n_obs}")
print(f"Clusters in view: {adata_ann.obs[CLUSTER_KEY].nunique()}")

In [ ]:
props_to_show = ["molecular_weight", "logp", "tpsa", "hbd", "rotatable_bonds",
                 "num_rings", "fraction_sp3", "n_bioactivities", "n_targets"]

fig, axes = plt.subplots(3, 3, figsize=(18, 14))
for ax, prop in zip(axes.flat, props_to_show):
    data = adata_ann.obs[[CLUSTER_KEY, prop]].dropna()
    top_clusters = data[CLUSTER_KEY].value_counts().head(8).index
    data = data[data[CLUSTER_KEY].isin(top_clusters)]
    if len(data) == 0:
        ax.set_visible(False)
        continue
    order = sorted(top_clusters, key=lambda x: int(x))
    sns.boxplot(data=data, x=CLUSTER_KEY, y=prop, ax=ax, order=order,
                palette="Set2", fliersize=2)
    ax.set_title(prop.replace("_", " ").title(), fontsize=11)
    ax.set_xlabel("Leiden cluster")

plt.suptitle(f"Properties per Leiden cluster ({MODELS[REFERENCE_MODEL]})", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
cluster_means = adata_ann.obs.groupby(CLUSTER_KEY)[all_props].mean()
cluster_means = cluster_means.loc[cluster_means.index.isin(
    adata_ann.obs[CLUSTER_KEY].value_counts().head(10).index
)]

cluster_z = (cluster_means - cluster_means.mean()) / (cluster_means.std() + 1e-8)

fig, ax = plt.subplots(figsize=(14, 6))
sns.heatmap(
    cluster_z.T, cmap="RdBu_r", center=0, annot=True, fmt=".1f",
    ax=ax, linewidths=0.5, linecolor="white",
    yticklabels=[p.replace("_", " ").title() for p in cluster_z.columns],
)
ax.set_xlabel("Leiden cluster")
ax.set_title(f"Property z-scores per cluster ({MODELS[REFERENCE_MODEL]})", fontsize=13)
plt.tight_layout()
plt.show()

### UMAP colored by molecular properties

In [ ]:
epl.umap_feature_panel(
    adata, obsm_key=REFERENCE_MODEL,
    features=["molecular_weight", "logp", "tpsa", "num_rings", "n_targets", "n_bioactivities"],
    ncols=3,
    title=f"UMAP ({MODELS[REFERENCE_MODEL]}) colored by molecular properties",
)
plt.show()

### Comparing clustering across all models

Side-by-side UMAP of the reference model colored by Leiden clusters from
each model, to see how different embedding spaces partition the molecules.

In [ ]:
fig, axes = plt.subplots(1, len(MODELS), figsize=(5 * len(MODELS), 4.5))
ref_umap = adata.obsm[f"X_umap_{REFERENCE_MODEL}"]

for ax, (key, label) in zip(axes, MODELS.items()):
    cluster_key = f"leiden_{key}"
    clusters = adata.obs[cluster_key].astype(str)
    n_cl = clusters.nunique()
    palette = sns.color_palette("tab20", n_colors=n_cl)
    for i, cl in enumerate(sorted(clusters.unique(), key=lambda x: int(x))):
        mask = clusters == cl
        ax.scatter(ref_umap[mask, 0], ref_umap[mask, 1], s=3, alpha=0.5,
                   color=palette[i % len(palette)], label=cl)
    ax.set_title(f"Clusters from {label}\n({n_cl} clusters)", fontsize=10)
    ax.set_xticks([])
    ax.set_yticks([])

plt.suptitle(
    f"Reference UMAP ({MODELS[REFERENCE_MODEL]}) colored by Leiden clusters from each model",
    fontsize=13, y=1.02,
)
plt.tight_layout()
plt.show()

---
## 8. Detailed annotation examples

Inspect the full annotation output for individual molecules.

In [ ]:
import json

for ann in annotations[:3]:
    smi = ann["smiles"]
    print(f"\n{'=' * 80}")
    print(f"SMILES: {smi[:80]}{'...' if len(smi) > 80 else ''}")
    print(f"{'=' * 80}")

    pc = ann.get("physicochemical", {})
    if pc:
        print(f"  MW={pc.get('molecular_weight', '?'):.1f}  "
              f"LogP={pc.get('logp', '?'):.2f}  "
              f"TPSA={pc.get('tpsa', '?'):.1f}  "
              f"HBD={pc.get('hbd', '?')}  HBA={pc.get('hba', '?')}")

    targets = ann.get("targets", [])
    if targets:
        print(f"  Targets ({len(targets)}): {', '.join(t.get('target_name', '?') for t in targets[:5])}")

    moa = ann.get("mechanism_of_action", [])
    if moa:
        print(f"  MoA ({len(moa)}): {', '.join(m.get('mechanism_of_action', '?') for m in moa[:3])}")

    roles = ann.get("chebi_roles", [])
    if roles:
        print(f"  ChEBI roles ({len(roles)}): {', '.join(r.get('role', '?') for r in roles[:5])}")

    pathways = ann.get("kegg_pathways", [])
    if pathways:
        print(f"  KEGG pathways ({len(pathways)}): {', '.join(p.get('pathway_name', '?') for p in pathways[:5])}")

    diseases = ann.get("diseases", [])
    if diseases:
        print(f"  Diseases ({len(diseases)}): {', '.join(d.get('disease_name', '?') for d in diseases[:5])}")

    xrefs = ann.get("cross_references", {})
    if xrefs:
        print(f"  Cross-refs: PubChem CID={xrefs.get('pubchem_cid', '?')}, "
              f"ChEMBL={xrefs.get('chembl_id', '?')}")

---
## Summary

| What | How | Result |
|---|---|---|
| Generate embeddings | `BioEmbedder.embed_molecule(smiles, model)` | Vector per molecule |
| Load pre-computed | CSV -> AnnData `.obsm` | Multi-model comparison |
| Embedding heatmap | `epl.embedding_clustermap(adata, obsm_key)` | Internal structure |
| Similarity heatmap | `epl.plot_similarity_heatmap(adata, obsm_key)` | Pairwise similarity |
| Cross-model agreement | `epl.cross_model_similarity(adata, method=...)` | Cosine r / ARI / KNN |
| KNN overlap | `epl.knn_overlap(adata, obsm_keys)` | Neighborhood consistency |
| Leiden overview | `epl.leiden_overview(adata, obsm_key)` | Clusters + sizes |
| Feature-colored UMAP | `epl.umap_feature_panel(adata, obsm_key, features)` | Property gradients |
| Cluster enrichment | `epl.cluster_property_heatmap(adata, cluster_key, props)` | Property z-scores |
| Annotation | `MoleculeAnnotator.annotate(smiles)` | 6 data sources |